# Проверки `c_nazn` в `ods.scd1_z_main_docum`

Цели:

1. Посмотреть все уникальные варианты `c_nazn` за первую неделю июня.
2. Выделить варианты `c_nazn`, которые могут относиться к эквайрингу.

Период задается параметрами ниже (по умолчанию: с `2026-06-01` по `2026-06-07` включительно).

In [ ]:
import os
import pandas as pd
from rail_connectors.connection import connect

print('Imports loaded')

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры периода: первая неделя июня
week_start = '2026-06-01'
week_end_exclusive = '2026-06-08'  # c 01 по 07 июня включительно

# Параметры исполнения
mem_limit = '8g'
preview_limit = 500
load_all_unique = False
load_all_ekv = False

table_name = 'ods.scd1_z_main_docum'

# Правила нормализации для группировки
# 1) "по договору" НЕ режем, чтобы не терять важные эквайринговые формулировки.
# 2) "Межбанковское вознаграждение UnionPay" -> оставляем только эту фразу.
# 3) "Комиссия, уплаченная за обслуживание платежных карт (service fee) UnionPay" -> оставляем только эту фразу.
# 4) "%зачисление по массиву%" -> "зачисления пенсии".
#
# Важно: используем POSIX-класс [[:space:]] для Impala (устойчивее, чем \s).
unionpay_interbank_phrase_pattern = r'.*межбанковское[[:space:]]+вознаграждение[[:space:]]+unionpay.*'
unionpay_interbank_canonical = 'Межбанковское вознаграждение UnionPay'

unionpay_service_fee_phrase_pattern = (
    r'.*комиссия,?[[:space:]]*уплаченная[[:space:]]+за[[:space:]]+обслуживание[[:space:]]+платежных[[:space:]]+карт[[:space:]]*[(]service fee[)][[:space:]]*unionpay.*'
)
unionpay_service_fee_canonical = 'Комиссия, уплаченная за обслуживание платежных карт (service fee) UnionPay'

pension_mass_pattern = r'.*зачислени[ея][[:space:]]+по[[:space:]]+массив[ау].*'
pension_mass_canonical = 'зачисления пенсии'

print(f'Period: [{week_start}, {week_end_exclusive})')
print(f'table={table_name}')

In [ ]:
# Подключение к Impala (как в 01_07_acq_dash.ipynb)
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connection initialized')

## 1) Все уникальные варианты `c_nazn` за первую неделю июня

In [ ]:
sql_unique_stats = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when nazn_norm rlike '{pension_mass_pattern}'
                then '{pension_mass_canonical}'
            when nazn_norm rlike '{unionpay_interbank_phrase_pattern}'
                then '{unionpay_interbank_canonical}'
            when nazn_norm rlike '{unionpay_service_fee_phrase_pattern}'
                then '{unionpay_service_fee_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped
    from scoped
)
select
    count(*) as total_rows,
    count(distinct c_nazn_raw) as unique_raw_c_nazn_count,
    count(distinct c_nazn_grouped) as unique_grouped_c_nazn_count
from normalized
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    unique_stats_df = imp.fetch(sql_unique_stats)

unique_stats_df

In [ ]:
sql_unique_variants_base = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when nazn_norm rlike '{pension_mass_pattern}'
                then '{pension_mass_canonical}'
            when nazn_norm rlike '{unionpay_interbank_phrase_pattern}'
                then '{unionpay_interbank_canonical}'
            when nazn_norm rlike '{unionpay_service_fee_phrase_pattern}'
                then '{unionpay_service_fee_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped
    from scoped
)
select
    c_nazn_grouped as c_nazn,
    count(*) as cnt,
    count(distinct c_nazn_raw) as raw_variants_collapsed
from normalized
group by c_nazn_grouped
order by cnt desc
"""

sql_unique_variants = (
    sql_unique_variants_base
    if load_all_unique
    else sql_unique_variants_base + f'\nlimit {preview_limit}'
)

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    unique_variants_df = imp.fetch(sql_unique_variants)

print(f'Rows loaded: {len(unique_variants_df):,}')
unique_variants_df.head(50)

In [ ]:
# Диагностика правил: сверяем LIKE и RLIKE для быстрых проверок матчинга
sql_rule_diagnostics = f"""
with scoped as (
    select
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
)
select
    sum(case when nazn_norm like '%межбанковское вознаграждение unionpay%' then 1 else 0 end) as unionpay_interbank_like_hits,
    sum(case when nazn_norm rlike '{unionpay_interbank_phrase_pattern}' then 1 else 0 end) as unionpay_interbank_rlike_hits,
    sum(case when nazn_norm like '%комиссия, уплаченная за обслуживание платежных карт (service fee) unionpay%' then 1 else 0 end) as unionpay_service_fee_like_hits,
    sum(case when nazn_norm rlike '{unionpay_service_fee_phrase_pattern}' then 1 else 0 end) as unionpay_service_fee_rlike_hits,
    sum(case when nazn_norm like '%зачисление по массиву%' or nazn_norm like '%зачисления по массиву%' then 1 else 0 end) as pension_mass_like_hits,
    sum(case when nazn_norm rlike '{pension_mass_pattern}' then 1 else 0 end) as pension_mass_rlike_hits
from scoped
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    rule_diagnostics_df = imp.fetch(sql_rule_diagnostics)

rule_diagnostics_df

## 2) Варианты `c_nazn`, которые могут относиться к эквайрингу

In [ ]:
# Ловим слова от корня "эквайр" (эквайринг, эквайринга, эквайринговый и т.д.)
ekv_pattern = r'(^|[^а-яa-z0-9])(эквайр[а-я]*)([^а-яa-z0-9]|$)'

sql_ekv_stats = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when nazn_norm rlike '{pension_mass_pattern}'
                then '{pension_mass_canonical}'
            when nazn_norm rlike '{unionpay_interbank_phrase_pattern}'
                then '{unionpay_interbank_canonical}'
            when nazn_norm rlike '{unionpay_service_fee_phrase_pattern}'
                then '{unionpay_service_fee_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped
    from scoped
)
select
    count(*) as scoped_rows,
    sum(case when nazn_norm rlike '{ekv_pattern}' then 1 else 0 end) as matched_rows,
    count(distinct case when nazn_norm rlike '{ekv_pattern}' then c_nazn_grouped end) as matched_unique_grouped
from normalized
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    ekv_stats_df = imp.fetch(sql_ekv_stats)

ekv_stats_df

In [ ]:
sql_ekv_variants_base = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when nazn_norm rlike '{pension_mass_pattern}'
                then '{pension_mass_canonical}'
            when nazn_norm rlike '{unionpay_interbank_phrase_pattern}'
                then '{unionpay_interbank_canonical}'
            when nazn_norm rlike '{unionpay_service_fee_phrase_pattern}'
                then '{unionpay_service_fee_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped
    from scoped
)
select
    c_nazn_grouped as c_nazn,
    count(*) as cnt,
    count(distinct c_nazn_raw) as raw_variants_collapsed
from normalized
where nazn_norm rlike '{ekv_pattern}'
group by c_nazn_grouped
order by cnt desc
"""

sql_ekv_variants = (
    sql_ekv_variants_base
    if load_all_ekv
    else sql_ekv_variants_base + f'\nlimit {preview_limit}'
)

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    ekv_variants_df = imp.fetch(sql_ekv_variants)

print(f'Rows loaded: {len(ekv_variants_df):,}')
ekv_variants_df.head(100)

In [ ]:
# Необязательно: сохранить результаты в CSV
save_to_csv = False
unique_out_path = './c_nazn_unique_first_week_june.csv'
ekv_out_path = './c_nazn_ekv_first_week_june.csv'

if save_to_csv:
    unique_variants_df.to_csv(unique_out_path, index=False)
    ekv_variants_df.to_csv(ekv_out_path, index=False)
    print(f'Saved: {unique_out_path}')
    print(f'Saved: {ekv_out_path}')
else:
    print('save_to_csv=False, nothing was written.')